# Edit Flows

A masked diffusion model can generate tokens in any order, but it can only generate 
sequences of a **fixed length**:
It starts from a fully-masked sequence and iteratively unmasks tokens until the sequence is
complete. 
This can force the model into "unnatural" tasks -- for example, filling in the arguments
of a function call when the closing parentheses has already been unmasked. The model would be 
constrained to a fixed number of tokens for that span. 

**Edit Flows** (Havasi et al., 2025) introduces a model that generates tokens through **edit operations**
-- insertions, deletions, and substitutions -- thus supporting variable length generation.
Rather than denoising a fixed canvas of masked
tokens, the model starts from a sequence from a prior distribution (e.g., empty sequence) and iteratively applies edits to
produce a sample from the data distribution. 

In this notebook we implement the **insertion-only** special case: the source distribution is a
sequence containing only the boundary tokens `[BOS]` and `[EOS]`, and the model generates text
by inserting tokens between them.  This is the simplest meaningful instance of Edit Flows and
captures the key ideas.

## Setup

In [ ]:
import sys
import os
import math

# Ensure the workshop/ directory is on the path (for ef_data, mdlm_model)
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from ef_data import (
    Text8Dataset, _download_text8, decode,
    VOCAB, VOCAB_SIZE, MASK_ID, BOS_ID, EOS_ID, PAD_ID, NEG_INF,
    bits_per_char,
)
from mdlm_model import MDLMBackbone

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'Vocabulary size: {VOCAB_SIZE}')
print(f'Special tokens:  [MASK]={MASK_ID}  [BOS]={BOS_ID}  [EOS]={EOS_ID}  [PAD]={PAD_ID}')

## 1. Data

We use the same **text8** dataset as in the MDLM notebook — 100 million characters of
lower-cased Wikipedia text, chunked into non-overlapping fixed-length sequences.

Now each sequence is wrapped with `[BOS]` (beginning-of-sequence)
and `[EOS]` (end-of-sequence) boundary tokens.  These tokens are *never* masked or modified
during training; they serve as fixed anchor points that tell the model where the sequence
begins and ends, and define the slots between which new tokens may be inserted.

In [ ]:
CACHE_DIR   = '../artifacts/text8_standalone'
CONTENT_LEN = 62   # characters per sequence, excluding BOS and EOS
SEQ_LEN     = CONTENT_LEN + 2   # total length: BOS + content + EOS

splits     = _download_text8(CACHE_DIR)
train_ds   = Text8Dataset(splits['train'], CONTENT_LEN)
val_ds     = Text8Dataset(splits['val'],   CONTENT_LEN)

train_loader = DataLoader(train_ds, batch_size=1024, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=1024, shuffle=False,
                          num_workers=2, drop_last=False)

print(f'Train: {len(train_ds):,} sequences  |  Val: {len(val_ds):,} sequences')
print(f'Sequence length: {SEQ_LEN}  (BOS + {CONTENT_LEN} chars + EOS)')

# Build word vocabulary from training text (used for WER evaluation).
word_vocab = frozenset(splits['train'].split())
print(f'Word vocabulary: {len(word_vocab):,} unique words')
print()
print('Sample sequences:')
for i in range(5):
    print(f'  [{i}]  "{decode(train_ds[i])}"')

---
## 2. The Forward Process

We want a reverse process that *inserts* tokens, so we need a forward process that
*deletes* them.  The difficulty is that deletion is ambiguous: given a partially-deleted
$x_t$ and the target $x_1$, there is generally no unique correspondence between positions
in the two sequences, making it hard to define a tractable propability path in sequence space.

Edit Flows resolve this by working in an **augmented space** where the alignment is
always explicit.  Sequences $x_t$ are converted to a fixed-length
representation $z_t$ of length $L$ using a blank token $\varepsilon$ (= `[MASK]`).  These blanks
mark deletion/insertion sites, so that $z_t$ and $z_1$ stay aligned through edit operations.  

**Example.**

**x-space** — sequences without positional alignment:

| | sequence |
|---|---|
| $x_1$ | K I T T E N |
| $x_t$ | K T N |

**z-space** — fixed-length with explicit deletion sites:

| | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $z_1$ | K | I | T | T | E | N |
| $z_t$ | K | $\varepsilon$ | T | $\varepsilon$ | $\varepsilon$ | N |

In **$z$-space** the alignment is unambiguous: positions 2, 4, and 5 are blank, so the
reverse process knows exactly where to insert I, T, and E.  In **$x$-space** we only
see the pair $(x_1{=}\texttt{KITTEN},\; x_t{=}\texttt{KTN})$, and multiple alignments
are consistent with that observation.

### 2.1 Source and target: $z_0$ and $z_1$

For a clean sequence $x_1 = [\texttt{BOS},\, c_1,\, \ldots,\, c_{L-2},\, \texttt{EOS}]$:

$$z_1 = x_1, \qquad
z_0 = [\texttt{BOS},\, \underbrace{\varepsilon,\, \ldots,\, \varepsilon}_{L-2},\, \texttt{EOS}]$$

Every content slot in $z_0$ is a blank, corresponding to the empty source sequence.
Boundary tokens are identical in both and are never modified.  Each position $i$ defines
an alignment pair $(z_0^i, z_1^i)$: content positions have $z_0^i = \varepsilon$,
$z_1^i = c_i$, meaning the forward process deletes $c_i$ there and the reverse process
must insert it back.

In [ ]:
def get_z(x1: torch.Tensor):
    """
    Construct the source z0 and target z1 from a batch of clean sequences.

    z0 is fully masked except at BOS, EOS, and PAD positions (which are copied
    from x1). z1 is the clean sequence x1 itself.

    Args:
        x1: [B, L]  clean token ids (includes BOS and EOS)
    Returns:
        z0: [B, L]  source — masked content, BOS/EOS/PAD preserved
        z1: [B, L]  target = x1
    """
    z0 = torch.full_like(x1, MASK_ID)
    special = (x1 == BOS_ID) | (x1 == EOS_ID) | (x1 == PAD_ID)
    z0[special] = x1[special]
    return z0, x1


# Quick check
example = train_ds[42].unsqueeze(0)  # [1, SEQ_LEN]
z0, z1 = get_z(example)
print(f'x1  (clean):   "{decode(example[0])}"')
print(f'z0  (masked):  "{decode(z0[0])}"')
print(f'z1  (target):  "{decode(z1[0])}"')

### 2.2 Stochastic interpolation: $z_t$

Having aligned $z_0$ and $z_1$ position-by-position, the forward process is identical to
the token-wise mixture path used in MDLM: at time $t$, each position is independently
set to its $z_1$ value with probability $\kappa(t)$, and kept as $z_0$ (i.e., `[MASK]`)
otherwise:

$$p(z_t^i \mid z_0^i, z_1^i) = \kappa(t)\,\delta_{z_1^i}(z_t^i)
  + (1-\kappa(t))\,\delta_{z_0^i}(z_t^i)$$

We use a **cubic schedule** $\kappa(t) = t^3$, which starts slow (near $t=0$ most positions
remain masked) and accelerates towards $t=1$ where every position is revealed.  The slower
early phase gives the model more gradient signal in the heavily-masked regime, where the
task is hardest.  A linear schedule $\kappa(t) = t$ would reveal tokens at a uniform rate,
spending equal training time at every noise level; the cubic schedule deliberately
over-samples the high-noise regime.

At $t=0$: $z_t = z_0$ (only `[BOS]` and `[EOS]` visible).
At $t=1$: $z_t = z_1$ (the full clean sequence).

In [ ]:
def get_zt(z0: torch.Tensor, z1: torch.Tensor, kappa: torch.Tensor) -> torch.Tensor:
    """
    Sample z_t by stochastically inserting tokens from z1 into z0.

    Each masked position in z0 is revealed (set to z1's token) independently
    with probability kappa.  BOS/EOS positions are already non-masked in z0
    and are left unchanged.

    Args:
        z0:    [B, L]  source (masked content, BOS/EOS preserved)
        z1:    [B, L]  target (clean tokens)
        kappa: [B]     insertion probability (one value per sequence)
    Returns:
        zt:    [B, L]  interpolated sequence
    """
    inserted_mask = torch.rand(z1.shape, device=z1.device) < kappa[:, None]
    return torch.where(inserted_mask, z1, z0)


# Show forward process at increasing kappa values
print(f'Original:  "{decode(example[0])}"')
print()
t_vals = [0.2, 0.5, 0.7, 0.9, 1.0]
for t_val in t_vals:
    kappa = torch.tensor([t_val ** 3])   # kappa = t^3
    zt = get_zt(z0, z1, kappa)
    n_masked = (zt == MASK_ID).sum().item()
    print(f't={t_val:.1f}  kappa={kappa.item():.3f}  masked={n_masked:2d}:  "{decode(zt[0])}"')

### 2.3 Compact representation: $x_t$

Recall that the augmented sequences $z_t$ were introduced as a training device: the
alignment they encode is sampled during training but should *not* be visible to the
model.  If the model could see $z_t$ directly — including the exact positions of the
`[MASK]` blanks — it would know precisely where each token needs to be inserted and
would not need to learn that from the data.  More importantly, it would be solving the
wrong problem: at inference time there is no $z_t$, only the sequence of tokens inserted
so far.

We therefore strip all `[MASK]` tokens from $z_t$ to obtain the compact sequence $x_t$,
which is what the model actually receives:

$$x_t = \texttt{remove\_masks}(z_t)$$

Because different sequences in a batch will have different numbers of revealed tokens, we
right-pad $x_t$ with `[PAD]` to a uniform length.

Denoising $x_t$ (i.e., predicting $x_1$) requires the model to (1) figure out *where* to insert tokens in
the current token context, and (2) implicitly marginalise over the valid
underlying $z_t, z_1$ paths. 

In [ ]:
def get_xt(zt: torch.Tensor) -> torch.Tensor:
    """
    Compact z_t by removing all [MASK] tokens and padding to a uniform length.

    The model operates on the compact sequence: only the tokens that have been
    inserted so far are passed in.  Padding ([PAD]) fills the remainder of the
    tensor so the batch can be stacked.

    Args:
        zt: [B, Lz]  interpolated sequences (may contain MASK_ID)
    Returns:
        xt: [B, Lx]  compact sequences, MASK tokens removed, padded with PAD_ID
    """
    non_mask = (zt != MASK_ID)
    max_len  = non_mask.sum(dim=1).max().item()
    B        = zt.shape[0]
    out      = torch.full((B, max_len), PAD_ID, dtype=zt.dtype, device=zt.device)
    for i in range(B):
        tokens = zt[i][non_mask[i]]
        out[i, :tokens.shape[0]] = tokens
    return out


# Show compact representation
print(f'z1 (full clean):    "{decode(z1[0])}"')
print()
for t_val in [0.3, 0.6, 1.0]:
    kappa = torch.tensor([t_val ** 3])
    zt = get_zt(z0, z1, kappa)
    xt = get_xt(zt)
    n_tok = (xt != PAD_ID).sum().item()
    print(f't={t_val:.1f}  kappa={kappa.item():.3f}  len={n_tok:2d}:  "{decode(xt[0])}"')

### 2.4 Insertion rates

The reverse process is a CTMC whose transitions are edit operations.  For our
insertion-only model, the rate of transitioning from the current sequence $x$ to a new
sequence obtained by inserting token $a$ after position $i$ is:

$$u^\theta_t\!\bigl(\mathrm{ins}(x, i, a)\bigm| x\bigr)
  = \lambda_i(x,t)\; Q(a \mid x, i, t)$$

- $\lambda_i(x, t) \geq 0$ is the **insertion rate** at position $i$: a non-negative
  scalar controlling how rapidly the model inserts a token after position $i$.
- $Q(a \mid x, i, t)$ is the **token distribution**: a categorical distribution over
  the vocabulary giving the probability of inserting token $a$ specifically.

Intuitively, over a short time interval $h$, an insertion fires at position $i$ with
probability $\approx h\,\lambda_i$, and the inserted token is drawn from $Q(\cdot \mid x, i)$.
A well-trained model should have high $\lambda_i$ only where a token is missing, and $Q$
concentrated on the correct token.

The `forward` method in §3 implements this decomposition for our insertion-only model.
See §3.1 of Havasi et al. (2025) for the full CTMC formulation including deletions and
substitutions.

---
## 3. Model Architecture

We use the same **DiT** (Diffusion Transformer) backbone as in the MDLM notebook: a
bidirectional (non-causal) transformer where each block uses Adaptive Layer Norm (AdaLN)
to condition on the timestep $t$ — the timestep embedding shifts and scales the layer
norms, so the model's behaviour changes smoothly with $t$.  Readers unfamiliar with this
architecture should refer to the MDLM notebook for a full walkthrough; here we focus on
the parts that differ.  The backbone maps a compact token sequence $x_t$ and a timestep
$t \in [0,1]$ to one logit vector per position.

The key difference from MDLM lies in how we interpret those logits.  In MDLM the full
softmax is a token distribution.  Here we **split the output** into two heads using
*the same* output tensor, requiring no additional parameters:

- **Insertion rate $\lambda$**: we re-purpose the `[MASK]` channel (index `MASK_ID`) as
  a scalar rate head.  Applying `softplus` ensures $\lambda \geq 0$.  Intuitively,
  $\lambda_j$ is "how urgently a new token should be inserted after position $j$."

- **Token distribution $Q$**: the remaining channels (all non-`[MASK]`, non-`[PAD]`
  indices) are passed through `log_softmax` to give a log-probability distribution over
  vocabulary tokens.  This answers the question "if we do insert here, *what* token?"

The `forward` method returns `(rate, log_rate, log_Q)`, zeroing out rate at `[PAD]`
positions so padding does not contribute to the loss.

In [ ]:
HIDDEN_SIZE = 128
N_BLOCKS    = 5
N_HEADS     = 8
COND_DIM    = 128
DROPOUT     = 0.1

backbone = MDLMBackbone(
    vocab_size=VOCAB_SIZE,
    hidden_size=HIDDEN_SIZE,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    cond_dim=COND_DIM,
    seq_len=SEQ_LEN,
    dropout=DROPOUT,
)
n_params = sum(p.numel() for p in backbone.parameters())
print(f'MDLMBackbone parameters: {n_params:,}')

# Demonstrate the rate / log_Q split on a small batch
x_test = train_ds[:4]
t_test = torch.rand(4)
logits = backbone(x_test, t_test)   # [B, L, vocab_size]

# Rate: softplus of the MASK channel
rate_demo = F.softplus(logits[:, :, MASK_ID])
print(f'rate  shape: {tuple(rate_demo.shape)}   min={rate_demo.min():.3f}  max={rate_demo.max():.3f}')

# Token distribution: log_softmax with MASK/PAD excluded
logits_q = logits.clone()
logits_q[:, :, MASK_ID] = NEG_INF
logits_q[:, :, PAD_ID]  = NEG_INF
log_Q_demo = logits_q.log_softmax(-1)
print(f'log_Q shape: {tuple(log_Q_demo.shape)}   (MASK/PAD channels = -inf)')

---
## 4. The Training Objective

Edit Flows are trained using the **Discrete Flow Matching** objective applied in the
augmented space of aligned sequences.  For our insertion-only model, the loss has two
terms that pull in opposite directions:

$$\mathcal{L}(\theta) = \mathbb{E}_{t,\, z_t}\!\left[
  \underbrace{\sum_{j}\, \lambda_j(x_t)}_{\text{rate penalty}}
  \;-\;
  \frac{\dot\kappa}{1 - \kappa}
  \sum_{\substack{i:\, z_t^i = \varepsilon}} \!\!
  \underbrace{\bigl(\log \lambda_{j(i)}(x_t) + \log Q(z_1^i \mid x_t, j(i))\bigr)}_{\text{log insertion rate for the correct token}}
\right]$$

**Rate penalty** (`rate.sum(dim=1)`): the sum of insertion rates over all positions in
$x_t$.  Without it, the model could drive the second term to $-\infty$ by
making all rates arbitrarily large. This term can be viewed as a penalty on the rates, so at
convergence the model should have *low* rate at positions that do not require insertions
and *high* rate only where tokens are still missing.

**Log insertion rate** (the second term): for each position $i$ in $z_t$ that is still
masked (i.e., $z_t^i = \varepsilon$), we reward the model for assigning high probability
to the correct insertion $z_1^i$.  The rate of inserting a specific token $a$ at position
$j$ is $\lambda_j \cdot Q(a \mid j)$, so its log is $\log\lambda_j + \log Q(a \mid j)$.

### The indexing trick

There is one subtlety: the loss sums over positions in $z_t$ (augmented space, length
$L_z$), but the model outputs are indexed by positions in $x_t$ (compact space, length
$L_x \leq L_z$).  We need to map each masked position $i$ in $z_t$ to the corresponding
position in $x_t$ — specifically, the position *just before* where the insertion would go,
so we can read off the right $\lambda$ and $Q$ values.

The number of non-mask tokens before position $i$ is `(~masked)[:i].sum()`, which equals
`(~masked).cumsum(dim=1)[i] - 1`.  A single vectorised cumsum gives the mapping for all
positions at once:

```python
xt_index = (~masked).long().cumsum(dim=1) - 1   # [B, Lz]
```

For example, given $z_t = [\texttt{BOS}, \_, a, \_, b, \texttt{EOS}]$ (where `_` = MASK):

| pos $i$ | $z_t^i$ | non-mask cumsum | `xt_index[i]` | meaning |
|---------|---------|----------------|---------------|---------|
| 0 | BOS | 1 | 0 | → $x_t[0]$ = BOS |
| 1 | MASK | 1 | 0 | insert after $x_t[0]$ = BOS |
| 2 | $a$ | 2 | 1 | → $x_t[1]$ = $a$ |
| 3 | MASK | 2 | 1 | insert after $x_t[1]$ = $a$ |
| 4 | $b$ | 3 | 2 | → $x_t[2]$ = $b$ |
| 5 | EOS | 4 | 3 | → $x_t[3]$ = EOS |

Continuing the example ($z_t = [\texttt{BOS}, \_, a, \_, b, \texttt{EOS}]$, single
sequence, batch index 0), `xt_index = [0, 0, 1, 1, 2, 3]`.  The gather at position
$i=1$ (first MASK) reads `log_rate[0, 0]` — the model's log-rate for inserting after
BOS — and `log_Q[0, 0, z_1^1]` — the log-probability of the correct token at that
insertion site.  The gather at position $i=3$ (second MASK) reads from $x_t[1] = a$,
giving the rate for inserting after $a$.

In [ ]:
def compute_ef_loss(model, x1: torch.Tensor, sampling_eps: float = 1e-5) -> torch.Tensor:
    """
    EditFlow training loss (rate penalty + weighted log insertion rate).
    See the markdown cell above for a full derivation and the indexing walkthrough.

    Args:
        model:        EditFlow instance (must expose get_z, get_zt, get_xt, forward)
        x1:           [B, L]  clean token ids (with BOS and EOS)
        sampling_eps: numerical clamp for t
    Returns:
        scalar loss
    """
    B = x1.shape[0]

    # 1. Sample t (stratified): divide [0,1] into B buckets, one sample per bucket
    eps_t  = torch.rand(B, device=x1.device)
    offset = torch.arange(B, device=x1.device) / B
    t      = (eps_t / B + offset).clamp(min=sampling_eps, max=1 - sampling_eps)
    kappa  = t ** 3           # [B]
    dkappa = 3.0 * t ** 2    # [B]

    # 2-4. Forward process
    z0, z1  = model.get_z(x1)
    zt      = model.get_zt(z0, z1, kappa)
    xt      = model.get_xt(zt)

    # 5. Model forward
    rate, log_rate, log_Q = model.forward(xt, t)

    # 6. Rate penalty: total insertion rate over all (non-pad) positions
    loss = rate.sum(dim=1)   # [B]

    # 7. Map each position in zt to its index in xt.
    #    For a masked position j, the index is that of the previous non-mask token
    #    (i.e., the position in xt right before where j would be inserted).
    masked   = (zt == MASK_ID)                             # [B, Lz]
    xt_index = (~masked).long().cumsum(dim=1) - 1         # [B, Lz]

    Lz        = zt.shape[1]
    batch_idx = torch.arange(B, device=zt.device)[:, None].expand(-1, Lz)
    log_rate_z = log_rate[batch_idx, xt_index]            # [B, Lz]
    log_Q_z1   = log_Q[batch_idx, xt_index, z1]          # [B, Lz]

    # 8. Token log-rate at masked positions; apply ELBO weight
    log_token_rates = (log_rate_z + log_Q_z1) * masked.float()   # [B, Lz]
    weight          = -dkappa / (1.0 - kappa)                    # [B]
    loss            = loss + weight * log_token_rates.sum(dim=1)  # [B]

    # 9. Normalise by masked-token count
    num_masked = masked.sum().clamp(min=1.0)
    return loss.sum() / num_masked

In [ ]:
class EditFlow(nn.Module):
    """
    EditFlow: insertion-only continuous-time flow matching for text.

    Bundles:
      - MDLMBackbone    (the transformer)
      - get_z / get_zt / get_xt   (forward process)
      - forward         (extracts rate + log_Q from backbone logits)
      - loss            (training objective — see compute_ef_loss above)
    """

    def __init__(self, vocab_size, hidden_size, n_blocks, n_heads,
                 cond_dim, seq_len, dropout=0.1):
        super().__init__()
        self.seq_len  = seq_len
        self.backbone = MDLMBackbone(vocab_size, hidden_size, n_blocks,
                                     n_heads, cond_dim, seq_len, dropout)

    # Forward-process helpers (defined as module-level functions above)
    get_z  = staticmethod(get_z)
    get_zt = staticmethod(get_zt)
    get_xt = staticmethod(get_xt)

    # ------------------------------------------------------------------
    # Model forward
    # ------------------------------------------------------------------

    def forward(self, xt: torch.Tensor, t: torch.Tensor):
        """
        Run the backbone and extract rate + log_Q.

        The backbone produces one logit per (position, token).  We re-purpose
        the MASK_ID channel as the insertion-rate head and the remaining
        channels as the token-distribution head.

        Args:
            xt:  [B, L]  compact token ids (no [MASK], may have [PAD])
            t:   [B]     timesteps in [0, 1]
        Returns:
            rate:     [B, L]  insertion rate (non-negative; zero at PAD)
            log_rate: [B, L]  log(rate), clamped for numerical stability
            log_Q:    [B, L, vocab_size]  log token distribution (MASK/PAD → -inf)
        """
        seq_lens = (xt != PAD_ID).sum(dim=1)
        logits   = self.backbone(xt, t, seq_lens=seq_lens)   # [B, L, vocab_size]
        pad_mask = (xt == PAD_ID)

        # Insertion rate: softplus of the MASK_ID logit channel
        rate     = F.softplus(logits[:, :, MASK_ID])
        rate     = rate.masked_fill(pad_mask, 0.0)
        log_rate = rate.clamp(min=1e-8).log()

        # Token distribution: log_softmax, MASK and PAD channels excluded.
        # Clone before modifying so rate's gradient path is not invalidated.
        logits_q = logits.clone()
        logits_q[:, :, MASK_ID] = NEG_INF
        logits_q[:, :, PAD_ID]  = NEG_INF
        log_Q = logits_q.log_softmax(-1)

        return rate, log_rate, log_Q

    # ------------------------------------------------------------------
    # Training objective
    # ------------------------------------------------------------------

    def loss(self, x1: torch.Tensor, sampling_eps: float = 1e-5) -> torch.Tensor:
        """
        EditFlow training loss.
        See compute_ef_loss() above for a detailed step-by-step walkthrough.
        """
        return compute_ef_loss(self, x1, sampling_eps)

---
## 5. Training

We instantiate `EditFlow` and verify the initial loss before training.  The class binds
the module-level `get_z`, `get_zt`, and `get_xt` as static methods (so `model.get_z`
and the bare `get_z` call the same code), and delegates `loss` to `compute_ef_loss`.

In [ ]:
# Instantiate the model
model = EditFlow(
    vocab_size=VOCAB_SIZE,
    hidden_size=HIDDEN_SIZE,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    cond_dim=COND_DIM,
    seq_len=SEQ_LEN,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')

loss_init = model.loss(train_ds[:64].to(device))
print(f'Initial loss: {loss_init.item():.3f}  ({bits_per_char(loss_init.item()):.3f} bpd)')

---
## 6. Training Loop

We use the same training recipe as MDLM: **AdamW** with a linear warmup and gradient
clipping (max norm 1.0).  The rate penalty term is always positive and has no lower bound
early in training, so the initial loss can be large; linear warmup prevents the optimizer
from making destructively large steps before the model has settled.

In [ ]:
def train_epoch(model, loader, optimizer, device, warmup_steps, step, base_lr):
    """Run one epoch. Returns (avg_loss_nats, updated_step)."""
    model.train()
    total_loss, n = 0.0, 0
    pbar = tqdm(loader, desc='train', leave=False)
    for x in pbar:
        x = x.to(device)
        # Linear warmup
        lr_scale = min(1.0, step / max(warmup_steps, 1))
        for pg in optimizer.param_groups:
            pg['lr'] = base_lr * lr_scale

        optimizer.zero_grad()
        loss = model.loss(x)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        n    += 1
        step += 1
        pbar.set_postfix(loss=f'{total_loss / n:.4f}')
    pbar.close()
    return total_loss / max(n, 1), step


@torch.no_grad()
def evaluate(model, loader, device, max_batches=50):
    """Estimate validation loss on a subset of the val set."""
    model.eval()
    total_loss, n = 0.0, 0
    for i, x in enumerate(loader):
        if i >= max_batches:
            break
        total_loss += model.loss(x.to(device)).item()
        n += 1
    return total_loss / max(n, 1)

In [ ]:
BASE_LR      = 3e-4
WARMUP_STEPS = 500
N_EPOCHS     = 20
CKPT_PATH    = '../artifacts/ef_workshop_best.pt'

os.makedirs(os.path.dirname(os.path.abspath(CKPT_PATH)), exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=0.01)
history   = {'train': [], 'val': []}
best_val  = float('inf')
step      = 0
start_epoch = 1

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model'])
    best_val    = ckpt['val_bpd'] / math.log2(math.e)   # convert bpd → nats to match best_val scale
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from checkpoint: epoch {ckpt["epoch"]}  val bpd={ckpt["val_bpd"]:.3f}')

for epoch in range(start_epoch, N_EPOCHS + 1):
    train_loss, step = train_epoch(model, train_loader, optimizer,
                                   device, WARMUP_STEPS, step, BASE_LR)
    val_loss = evaluate(model, val_loader, device)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    print(f'Epoch {epoch:3d}/{N_EPOCHS}  |  '
          f'train {train_loss:.4f} ({bits_per_char(train_loss):.3f} bpd)  |  '
          f'val {val_loss:.4f} ({bits_per_char(val_loss):.3f} bpd)')

    if val_loss < best_val:
        best_val = val_loss
        torch.save({'model': model.state_dict(), 'epoch': epoch,
                    'val_bpd': bits_per_char(val_loss)}, CKPT_PATH)
        print(f'             checkpoint saved  (val bpd = {bits_per_char(val_loss):.3f})')

In [ ]:
# Plot the training curve
fig, ax = plt.subplots(figsize=(8, 4))
epochs = list(range(1, len(history['train']) + 1))
ax.plot(epochs, [bits_per_char(l) for l in history['train']], label='Train', color='steelblue')
ax.plot(epochs, [bits_per_char(l) for l in history['val']],   label='Val',   color='crimson')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (bpd)')
ax.set_title('EditFlow Training Curve on text8')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Best val BPD: {bits_per_char(best_val):.3f}')

---
## 7. Sampling

Sampling reverses the forward process: we start from the prior
$x_0 = [\texttt{BOS}, \texttt{EOS}]$ and integrate the CTMC forward in time from
$t = 0$ to $t = 1$, inserting tokens at each step.

We use a first-order **Euler approximation** of the CTMC (Equation 1 in the paper).
With step size $h = 1 / N_\text{steps}$ and current state $x_t$:

1. **Forward pass** → insertion rate $\lambda_j(x_t)$ and token distribution
   $Q(\cdot \mid x_t, j)$ at each position $j$.
2. **Insertion decisions**: position $j$ triggers an insertion with probability
   $\min(h\,\lambda_j,\, 1)$.  Sample $\text{Bernoulli}(h\,\lambda_j)$ independently at
   every position.
3. **Token identity**: for each position where an insertion fires, sample the token from
   $Q(\cdot \mid x_t, j)$.
4. **Constraints**:
   - No insertions at `[EOS]` or `[PAD]` positions (the sequence can only grow to the left
     of EOS, not beyond it).
   - Cap total insertions per sample to stay within `seq_len` (budget constraint).
5. **Interleave**: scatter existing tokens and newly-inserted tokens into a new,
   longer tensor using cumulative index offsets.

Each Euler step grows the sequence by however many insertions fired.  After $N_\text{steps}$
steps the model has built up the full sequence from the empty prior.  More steps give
smoother trajectories and better samples, at the cost of more forward passes.

In [ ]:
@torch.no_grad()
def _ef_sample(self, n_samples: int, num_steps: int, device: torch.device,
               return_trajectory: bool = False):
    """
    Generate sequences via Euler integration of the EditFlow insertion process.

    We start from the prior p(z_0): a sequence containing only [BOS] and [EOS].
    At each Euler step we stochastically insert new tokens, growing the sequence
    until t = 1.

    Args:
        n_samples:          number of sequences to generate
        num_steps:          number of Euler steps
        device:             torch device
        return_trajectory:  if True, also return list of intermediate states
    """
    # Prior: [BOS, EOS]
    xt = torch.zeros((n_samples, 2), dtype=torch.long, device=device)
    xt[:, 0] = BOS_ID
    xt[:, 1] = EOS_ID

    trajectory = [xt.clone()] if return_trajectory else None
    h          = 1.0 / num_steps   # Euler step size
    ts         = torch.linspace(0, 1, num_steps + 1, device=device)[:-1]

    for t_val in ts:
        # (a) Broadcast t to shape [B]
        t = t_val.expand(n_samples)

        # (b) Forward pass: rate λ and token distribution Q at each position
        rate, log_rate, log_Q = self.forward(xt, t)

        # (c) Insertion probability per position (Euler first-order approximation)
        insert_prob = (h * rate).clamp(max=1.0)
        do_insert   = torch.rand_like(insert_prob) < insert_prob

        # (d) Sample the token to insert at each candidate position
        new_tokens = torch.distributions.Categorical(logits=log_Q).sample()

        # (e) Suppress insertion at EOS and PAD positions
        eos_or_pad = (xt == EOS_ID) | (xt == PAD_ID)
        do_insert  = do_insert & ~eos_or_pad

        # (f) Cap insertions per sample to stay within seq_len
        seq_lens  = (xt != PAD_ID).sum(dim=1)
        budget    = (self.seq_len - seq_lens).clamp(min=0)
        cum_ins   = do_insert.long().cumsum(dim=1)
        do_insert = do_insert & (cum_ins <= budget[:, None])

        if not do_insert.any():
            if return_trajectory:
                trajectory.append(xt.clone())
            continue

        # (g) Scatter: interleave existing tokens with new insertions.
        #   contrib[j] = 1 (existing token only) or 2 (existing + one insertion after it).
        #   out_idx[j] = cumsum(contrib)[j] - contrib[j] = output position for token j.
        #   Example: xt=[BOS,a,EOS], insert after 'a':
        #     contrib  = [1, 2, 1]
        #     out_idx  = [0, 1, 3]   → BOS→0, a→1, EOS→3
        #     ins_idx  = [1, 2, 4]   → inserted token → 2  (between a and EOS)
        #   When adjacent positions both fire, contrib can be [2, 2, ...], and the cumsum
        #   still assigns non-overlapping output indices because each insertion adds exactly
        #   1 to all subsequent positions.
        non_pad  = (xt != PAD_ID)
        contrib  = non_pad.long() + do_insert.long()
        out_idx  = contrib.cumsum(dim=1) - contrib

        B_cur    = xt.shape[0]
        new_L    = contrib.sum(dim=1).max().item()
        new_xt   = torch.full((B_cur, new_L), PAD_ID, dtype=xt.dtype, device=device)

        batch_idx             = torch.arange(B_cur, device=device)[:, None].expand_as(xt)
        new_xt[batch_idx[non_pad], out_idx[non_pad]]  = xt[non_pad]
        ins_idx               = out_idx + 1
        new_xt[batch_idx[do_insert], ins_idx[do_insert]] = new_tokens[do_insert]

        xt = new_xt

        if return_trajectory:
            trajectory.append(xt.clone())

    return (xt, trajectory) if return_trajectory else xt


# Attach sample to the class here rather than inside the class definition, so that
# the full training loop (Section 6) can run first and the reader sees a working model
# before the sampling code is introduced.
EditFlow.sample = _ef_sample

---
## 8. Results and Evaluation

In [ ]:
# Visualize the insertion trajectory for a single generated sequence
model.eval()
NUM_STEPS = SEQ_LEN * 2   # more steps → smoother insertion

_, trajectory = model.sample(n_samples=1, num_steps=NUM_STEPS,
                              device=device, return_trajectory=True)

snap_steps = [0, NUM_STEPS // 8, NUM_STEPS // 4, NUM_STEPS // 2,
              3 * NUM_STEPS // 4, NUM_STEPS]
print('Insertion trajectory (one generated sequence):')
print(f'  {"Step":>4}   t≈    tokens   Sequence')
print('  ' + '-' * 72)
for s in snap_steps:
    state   = trajectory[min(s, len(trajectory) - 1)]
    n_tok   = int((state[0] != PAD_ID).sum().item())
    seq     = decode(state[0])
    t_approx = s / NUM_STEPS
    print(f'  {s:>4}  {t_approx:.2f}     {n_tok:3d}    "{seq}"')

In [ ]:
# Generate a batch of samples and compare to real val sequences
N_DISPLAY = 8
samples   = model.sample(n_samples=N_DISPLAY, num_steps=NUM_STEPS, device=device)

print('Generated samples:')
print('-' * 70)
for i in range(N_DISPLAY):
    print(f'  [{i}]  "{decode(samples[i])}"')

print()
print('Reference (val) sequences:')
print('-' * 70)
for i in range(N_DISPLAY):
    print(f'  [{i}]  "{decode(val_ds[i])}"')

### Word Error Rate (WER)
WER measures the fraction of characters in generated text that belong to words **not**
found in the training vocabulary.  A lower WER indicates that the model produces more
plausible English words.  This metric is identical to the one used in the MDLM notebook.

### Number of sampling steps
More Euler steps give smoother trajectories and better samples, but require more forward
passes.  We use a fixed `num_steps=128` for evaluation.

In [ ]:
# WER: fraction of characters in words absent from the training vocabulary
def word_error_rate(samples, vocab):
    """Mean per-sequence WER. Empty sequences and all-unknown sequences score 1.0."""
    wers = []
    for text in samples:
        words = text.split()
        if not words:
            wers.append(1.0)
            continue
        total   = sum(len(w) for w in words)
        invalid = sum(len(w) for w in words if w not in vocab)
        wers.append(invalid / total)
    return sum(wers) / len(wers) if wers else 1.0


print('Computing WER on 100 generated sequences ...')
N_WER    = 100
gen_seqs = model.sample(n_samples=N_WER, num_steps=NUM_STEPS, device=device)
hyps     = [decode(s) for s in gen_seqs]
wer      = word_error_rate(hyps, word_vocab)
print(f'WER (chars in unknown words): {wer:.3f}')
print()
print('Example generated sequences:')
for h in hyps[:5]:
    print(f'  "{h}"')